# wag 🐾 — preflight

two measurements to take **before** committing to a 4–6 hour training run, because both
of them can change what that run should be.

1. **what does base Qwen3.5-4B actually score?** v1's numbers belong to a 2B. base 2B
   scored 1.60 on helpfulness; the 4B will score higher and nobody knows how much. every
   "v2 improved X by Y" claim is meaningless until this exists.
2. **does LoRA hold the bare-prompt voice?** v1's best single result was that the voice
   survived an *empty* system prompt — 3.73 voice with no prompt against 3.71 with one.
   that means the personality went into the weights rather than riding on the prompt.
   nobody has checked whether a rank-32 adapter holds that as well as a full fine-tune,
   and the answer decides open question #2: LoRA, or pay for an 80 GB card.

the probe trains on **v1's existing 1,564 rows**, which are already in the repo. so this
notebook needs no v2 data, no gemini credit, and nothing that's currently blocked. it's
about 40 minutes on an A100.

there's also a memory check first, because the handoff's full-FT projection (~35 GB
before activations) is arithmetic, not a measurement, and it's the number the whole
LoRA decision rests on.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# same pins as train.ipynb. 5.16 doesn't exist and 6.0 will break the qwen3_5 arch;
# peft for the adapter, datasets for the loader
!pip install -q -U "transformers>=5.15,<6" "peft>=0.14" "datasets>=3" "accelerate>=1.2"

In [ ]:
import json, os, re, sys, time
from pathlib import Path

import torch
from google.colab import drive

drive.mount("/content/drive")

BASE     = "Qwen/Qwen3.5-4B"
SEQ_LEN  = 1024
DRIVE    = Path("/content/drive/MyDrive/wag")
REPO     = Path("/content/wag")
OUT      = DRIVE / "preflight"
OUT.mkdir(parents=True, exist_ok=True)

if not REPO.exists():
    !git clone -q https://github.com/Metrix187/wag {REPO}
sys.path.insert(0, str(REPO))

gpu = torch.cuda.get_device_name(0)
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"gpu: {gpu}  ({total:.0f} GB)")
if "A100" not in gpu:
    print("!! the memory numbers below assume a 40GB A100 — read them relative to your card")

## 0 — memory, measured rather than projected

the handoff projects full fine-tuning the 4B at ~35 GB *before activations*, and notes
gradient checkpointing is unavailable on this architecture (the linear-attention layers
raise `CheckpointError` on recompute), so there's no lever to pull. that projection is
what rules out full FT on a 40 GB card. worth confirming it with real allocations.

In [ ]:
from transformers import AutoModelForImageTextToText, AutoModelForCausalLM

torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
t0 = time.time()
try:
    model = AutoModelForImageTextToText.from_pretrained(
        BASE, dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
except Exception as e:
    print(f"image-text loader failed ({e}); falling back to causal-lm")
    model = AutoModelForCausalLM.from_pretrained(
        BASE, dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
print(f"loaded in {time.time()-t0:.0f}s")

weights = torch.cuda.memory_allocated() / 1e9
n_all = sum(p.numel() for p in model.parameters())
vis = sum(p.numel() for n, p in model.named_parameters()
          if re.search(r"vision|visual|image_|video_|patch_embed", n, re.I))

print(f"\nparams total   {n_all/1e9:.2f}B   (vision {vis/1e9:.2f}B)")
print(f"weights on gpu {weights:.1f} GB")
print(f"\nprojected for a FULL fine-tune, from these weights:")
print(f"  weights        {weights:.1f} GB")
print(f"  grads          {(n_all-vis)*2/1e9:.1f} GB   (bf16, text params only)")
print(f"  adam moments   {(n_all-vis)*4/1e9:.1f} GB")
print(f"  ---> {weights + (n_all-vis)*6/1e9:.1f} GB before a single activation, "
      f"on a {total:.0f} GB card")
print("\nhandoff projected 34.9 GB. gradient checkpointing is NOT available here, so "
      "activations sit on top of that uncompressed.")

## 1 — base 4B baseline

the 20 held-out prompts plus the 10 multi-turn conversations, on the untouched base
model. this is the number every v2 claim gets compared against.

note base gets handed the wag system prompt. that's deliberate and it's how v1 measured
it — and it's also how base scored **4.03 on voice, above the fine-tune**, by emitting
emoji and saying nothing. the voice metric has since been fixed to require the reply to
contain information, so expect base to score a lot lower than 4.03 now. that drop is the
metric working, not the model changing.

In [ ]:
from transformers import AutoTokenizer
sys.path.insert(0, str(REPO))
from eval import EVAL_PROMPTS, MULTI_PROMPTS, _stop_ids, voice_score, convo_score
from gen_anchors import SYSTEM_PROMPT

tok = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
model.config.use_cache = True
model.eval()

def respond(messages, max_new=500):
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new, do_sample=True,
                             temperature=0.7, top_p=0.9,
                             eos_token_id=_stop_ids(tok),
                             pad_token_id=tok.pad_token_id or tok.eos_token_id)
    reply = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    return re.split(r"\n(?:user|assistant)\s*\n", reply)[0].strip()

def run_all(system, tag):
    single, multi = [], []
    for pid, cat, prompt in EVAL_PROMPTS:
        msgs = ([{"role": "system", "content": system}] if system else []) + \
               [{"role": "user", "content": prompt}]
        single.append({"id": pid, "category": cat, "prompt": prompt,
                       "response": respond(msgs)})
        print(".", end="", flush=True)
    for cid, cat, turns in MULTI_PROMPTS:
        msgs = [{"role": "system", "content": system}] if system else []
        ex = []
        for t in turns:
            msgs.append({"role": "user", "content": t})
            r = respond(msgs)
            msgs.append({"role": "assistant", "content": r})
            ex.append({"user": t, "reply": r})
        multi.append({"id": cid, "category": cat, "turns": ex})
        print("+", end="", flush=True)
    (OUT / f"{tag}_single.jsonl").write_text(
        "".join(json.dumps(r, ensure_ascii=False) + "\n" for r in single),
        encoding="utf-8")
    (OUT / f"{tag}_multi.jsonl").write_text(
        "".join(json.dumps(r, ensure_ascii=False) + "\n" for r in multi),
        encoding="utf-8")
    print(f"\nsaved {tag} -> {OUT}")
    return single, multi

def report(single, multi, label):
    v = sum(voice_score(r["response"])["score"] for r in single) / len(single)
    w = sum(voice_score(r["response"])["words"] for r in single) / len(single)
    cs = [convo_score(c["turns"]) for c in multi]
    leaks = sum(c["speaker_leaks"] for c in cs)
    narr = sum(c["narrates_user"] for c in cs)
    energy = sum(c["energy_violations"] for c in cs)
    cv = sum(c["voice"] for c in cs) / len(cs)
    print(f"\n=== {label} ===")
    print(f"  single-turn voice   {v:.2f} / 5     (mean {w:.0f} words)")
    print(f"  multi-turn voice    {cv:.2f} / 5")
    print(f"  turn-taking fails   {leaks} speaker leaks, {narr} narrating the user")
    print(f"  energy mismatches   {energy}")
    return {"voice": round(v, 2), "convo_voice": round(cv, 2), "words": round(w),
            "leaks": leaks, "narrates": narr, "energy": energy}

base_single, base_multi = run_all(SYSTEM_PROMPT, "base4b")
BASE_SCORES = report(base_single, base_multi, "base Qwen3.5-4B, wag prompt")

In [ ]:
# and with no system prompt at all — the other half of the baseline. if base already
# scores well here, a high bare-prompt score from the fine-tune proves less than it looks
bare_single, bare_multi = run_all(None, "base4b_nosys")
BASE_BARE = report(bare_single, bare_multi, "base Qwen3.5-4B, NO system prompt")

(OUT / "baseline.json").write_text(json.dumps(
    {"model": BASE, "with_prompt": BASE_SCORES, "no_prompt": BASE_BARE}, indent=1),
    encoding="utf-8")
print(f"\nbaseline written -> {OUT/'baseline.json'}")
print("keep this. every v2 number in the model card is a delta against it.")

## 2 — the LoRA probe

train a rank-32 adapter on v1's 1,564 rows, one epoch, then check whether the voice
survives an empty system prompt.

**v1's full fine-tune scored 3.73 bare / 3.71 prompted.** that near-equality is the
result worth protecting — it's what "the personality is in the weights" means. if the
adapter lands well under it, that's the signal to find an 80 GB card and do this
properly; if it holds, LoRA is free and the checkpoint problem evaporates.

one epoch on 1,564 rows is not v2's recipe. it doesn't need to be — this is asking
whether the *mechanism* works, not how good the model gets.

In [ ]:
del model
torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

rows = [json.loads(l) for l in (REPO / "data" / "train.jsonl").open(encoding="utf-8")]
print(f"{len(rows)} rows from v1")

IGNORE = -100

def encode(rec):
    msgs = rec["messages"]
    p = tok.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True)
    f = tok.apply_chat_template(msgs, tokenize=False)
    p_ids = tok(p, add_special_tokens=False)["input_ids"]
    f_ids = tok(f, add_special_tokens=False)["input_ids"]
    if len(f_ids) > SEQ_LEN:
        return None
    labels = [IGNORE] * len(p_ids) + f_ids[len(p_ids):]
    return {"input_ids": f_ids, "labels": labels[:len(f_ids)],
            "attention_mask": [1] * len(f_ids)}

encoded = [e for e in (encode(r) for r in rows) if e]
sup = sum(sum(1 for x in e["labels"] if x != IGNORE) for e in encoded)
tot = sum(len(e["labels"]) for e in encoded)
print(f"{len(encoded)} usable, supervised {100*sup/tot:.0f}% (want roughly 45-70%)")

In [ ]:
from dataclasses import dataclass
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments

ds = Dataset.from_list(encoded).train_test_split(test_size=0.02, seed=20260823)

@dataclass
class PadCollator:
    pad_id: int
    def __call__(self, feats):
        n = max(len(f["input_ids"]) for f in feats)
        out = {"input_ids": [], "labels": [], "attention_mask": []}
        for f in feats:
            gap = n - len(f["input_ids"])
            out["input_ids"].append(f["input_ids"] + [self.pad_id] * gap)
            out["labels"].append(f["labels"] + [IGNORE] * gap)
            out["attention_mask"].append(f["attention_mask"] + [0] * gap)
        return {k: torch.tensor(v) for k, v in out.items()}

try:
    model = AutoModelForImageTextToText.from_pretrained(
        BASE, dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
except Exception:
    model = AutoModelForCausalLM.from_pretrained(
        BASE, dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)

for n, p in model.named_parameters():
    if re.search(r"vision|visual|image_|video_|patch_embed", n, re.I):
        p.requires_grad = False

model = get_peft_model(model, LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
))
model.print_trainable_parameters()
model.config.use_cache = False

args = TrainingArguments(
    output_dir=str(OUT / "lora_probe"),
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,                  # lora rate, not 2e-5
    lr_scheduler_type="cosine",
    warmup_steps=0.05,                   # a float <1 is a ratio on v5
    logging_steps=10,
    save_strategy="no",                  # it's a probe, the adapter is disposable
    eval_strategy="steps", eval_steps=25,
    bf16=True,
    optim="adamw_torch",
    gradient_checkpointing=False,        # CheckpointError on this arch. do not turn on
    report_to="none", seed=20260823,
)
trainer = Trainer(model=model, args=args, train_dataset=ds["train"],
                  eval_dataset=ds["test"], data_collator=PadCollator(
                      tok.pad_token_id or tok.eos_token_id))
t0 = time.time()
trainer.train()
print(f"\ntrained in {(time.time()-t0)/60:.0f} min")
print(f"peak gpu during training: {torch.cuda.max_memory_allocated()/1e9:.1f} GB")

In [ ]:
model.config.use_cache = True
model.eval()

probe_single, probe_multi = run_all(SYSTEM_PROMPT, "lora_probe")
PROBE = report(probe_single, probe_multi, "LoRA probe, wag prompt")

probe_bare_s, probe_bare_m = run_all(None, "lora_probe_nosys")
PROBE_BARE = report(probe_bare_s, probe_bare_m, "LoRA probe, NO system prompt")

## the verdict

In [ ]:
V1_PROMPTED, V1_BARE = 3.71, 3.73      # v1, full fine-tune of the 2B

print(f"{'':28} {'prompted':>9} {'bare':>7} {'gap':>7}")
print("-" * 54)
print(f"{'v1 full FT (2B)':28} {V1_PROMPTED:9.2f} {V1_BARE:7.2f} "
      f"{V1_BARE-V1_PROMPTED:+7.2f}")
print(f"{'base 4B, untrained':28} {BASE_SCORES['voice']:9.2f} "
      f"{BASE_BARE['voice']:7.2f} {BASE_BARE['voice']-BASE_SCORES['voice']:+7.2f}")
print(f"{'LoRA probe (4B, 1 epoch)':28} {PROBE['voice']:9.2f} "
      f"{PROBE_BARE['voice']:7.2f} {PROBE_BARE['voice']-PROBE['voice']:+7.2f}")

gap = PROBE_BARE["voice"] - PROBE["voice"]
lift = PROBE_BARE["voice"] - BASE_BARE["voice"]
print()
if lift < 0.5:
    print("!! the adapter barely moved the bare-prompt voice off base. either the probe "
          "undertrained (one epoch, v1's data) or rank 32 isn't reaching it.")
    print("   -> try r=64 before concluding anything about LoRA in general.")
elif gap < -0.6:
    print("!! the voice is leaning on the system prompt: it drops sharply without one.")
    print("   v1's full fine-tune didn't do that. this is the signal to price an 80 GB "
          "card and do v2 as a full fine-tune.")
else:
    print("ok — the bare-prompt voice holds up without the prompt, the same way v1's "
          "full fine-tune did.")
    print("   -> LoRA is fine for v2. ~200 MB checkpoints, fits the 40 GB card, and "
          "the drive-space problem goes away.")

print("\nnumbers are one epoch on v1's data — read the SHAPE (does bare track prompted), "
      "not the absolute values.")
print("\nand go read out/ by hand. v1's best decisions all came from reading rows, not "
      "from trusting a mean.")